# 00 · Smoke test

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (float16) or **GPU L4** (bfloat16) — set `HW` below |
| Internet | **ON** (pip install, model download) |
| Attach | the code (Kaggle dataset `scopesum-code`) unless you set `REPO_URL`; optionally the Gemma 4 model from Kaggle Models if `model.source=local` |
| Expected runtime | ~10–15 min (most of it is the model download) |

What it checks:
1. unit tests pass;
2. the real Gemma 4 config (layer types, sliding window, KV sharing);
3. variant A through our wrapper == the library's own forward (same code path);
4. one forward pass per variant A–E with float16 NaN/inf checks, time and memory;
5. **mask sanity**: changing a token the sliding mask hides does not change the first local layer's attention output at the query; changing a visible token does. Global layers still see everything.

In [ ]:
# --- get the code: git clone (needs Internet) OR an attached Kaggle dataset ---
REPO_URL = "https://github.com/loaferct/scopesum.git"  # private repo: needs the Kaggle Secret GITHUB_TOKEN; "" = use an attached code dataset
CODE_DATASET_DIR = "/kaggle/input"  # searched for a folder containing scopesum/__init__.py
HW = "t4"  # "t4" (float16) or "l4" (bfloat16)

import glob, os, shutil, subprocess, sys
CODE_DIR = "/kaggle/working/scopesum"

def clone_repo(url, dest, secret_name="GITHUB_TOKEN"):
    """git clone; uses a GitHub token from Kaggle Secrets for private repos (never printed)."""
    auth_url = url
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(secret_name)
        auth_url = url.replace("https://", f"https://x-access-token:{token}@")
    except Exception as e:
        print(f"No Kaggle Secret {secret_name!r} ({type(e).__name__}); trying an anonymous clone")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, dest], capture_output=True, text=True)
    if r.returncode != 0:
        print("git clone failed:", r.stderr.replace(auth_url, url)[-500:])
        return False
    subprocess.run(["git", "-C", dest, "remote", "set-url", "origin", url])  # drop the token from .git/config
    print("cloned", url, subprocess.run(["git", "-C", dest, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
    return True

if not os.path.exists(os.path.join(CODE_DIR, "scopesum", "__init__.py")):
    if not (REPO_URL and clone_repo(REPO_URL, CODE_DIR)):
        hits = [h for h in glob.glob(CODE_DATASET_DIR + "/**/scopesum/__init__.py", recursive=True) if "/tests/" not in h]
        assert hits, "Attach the scopesum code dataset or set REPO_URL"
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE_DIR)
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print("code dir:", CODE_DIR)

In [ ]:
!pip uninstall -y -q torchao  # Kaggle ships torchao 0.10; peft>=0.21 refuses it and we do not use torchao
!pip install -q -r requirements.txt
!python -m pytest -q tests -p no:cacheprovider

In [ ]:
import json, time
import numpy as np
import torch, transformers, peft
from scopesum.config import load_config
from scopesum.data import build_example, load_tokenizer, resolve_model_path, resolve_sum_token_id
from scopesum.model import build_model, describe_attention_layout, isfinite_report

print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
# Build once with variant C (so the <sum> embedding exists); we switch lm.variant per test below.
cfg = load_config("configs/base.yaml", "configs/variant_C.yaml", f"configs/hw_{HW}.yaml")
path = resolve_model_path(cfg)
tok = load_tokenizer(cfg, path)
sum_id = resolve_sum_token_id(tok, cfg)
print("model path:", path, "| <sum> id:", sum_id)
lm = build_model(cfg, path, sum_id, for_training=False)
lm.eval()
layout = describe_attention_layout(lm.model)
print(json.dumps({k: v for k, v in layout.items() if k != "layer_types"}, indent=2))
print("layer types:", "".join("S" if t == "sliding_attention" else "F" for t in layout["layer_types"]))
print(f"GPU memory after load: {torch.cuda.memory_allocated() / 2**30:.2f} GB")

In [ ]:
# A synthetic file long enough (> sliding window) that the masks differ between variants.
def make_sample(n_funcs=24):
    parts = ["import math\n\n"]
    for i in range(n_funcs):
        parts.append(
            f"def func_{i}(a, b=1):\n"
            f"    \"\"\"Compute value {i}.\"\"\"\n"
            f"    total = a * {i} + b\n"
            f"    for k in range(3):\n"
            f"        total += math.sqrt(k + {i})\n"
            f"    return total\n\n"
        )
    parts.append("class Runner:\n    def run(self):\n        return func_0(1) + func_3(2)\n\nprint(Runner().run())\n")
    return "".join(parts)

SEQ_LEN = cfg.data.seq_len
text = make_sample()
enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
window = {"raw_ids": enc["input_ids"], "offsets": [list(o) for o in enc["offset_mapping"]], "text": text}
n_raw = len(window["raw_ids"])
print("raw tokens:", n_raw)
assert n_raw < SEQ_LEN - 150, "sample too long for seq_len; lower n_funcs"
K_SMOKE = 48  # any K works for the smoke test; real K comes from stats.json
MODE = {"A": "none", "B": "fixed", "C": "scope", "D": "scope", "E": "scope"}
examples = {v: build_example(window, MODE[v], tok.bos_token_id, sum_id, tok.pad_token_id, SEQ_LEN, chunk_size=K_SMOKE) for v in "ABCDE"}
for v, ex in examples.items():
    print(v, "len", ex.length, "| <sum>", sum(k == 2 for k in ex.kind), "| truncated", ex.n_truncated, "| edges", len(ex.seg_edges))

## Variant A == the library's default forward
LoRA `B` matrices start at zero, so the PEFT model equals the base model here. Expect a max |Δlogit| of 0 (or tiny if kernels differ).

In [ ]:
lm.variant = "A"
ex_a = build_example(window, "none", tok.bos_token_id, sum_id, tok.pad_token_id, n_raw + 1)  # no padding
batch = lm.make_batch([ex_a])
with torch.no_grad():
    ours_h = lm(batch, compute_loss=False)["hidden"]
    ours = torch.nn.functional.linear(ours_h[:, -64:], lm.lm_head.weight).float()
    if lm.softcap:
        ours = torch.tanh(ours / lm.softcap) * lm.softcap
    ref = lm.model(input_ids=batch["input_ids"].to(lm.device), attention_mask=batch["pad_mask"].to(lm.device),
                   use_cache=False, logits_to_keep=64).logits.float()
print("max |logit diff| (ours vs library):", (ours - ref).abs().max().item())
print("argmax agreement:", (ours.argmax(-1) == ref.argmax(-1)).float().mean().item())

## One forward pass per variant (+ float16 NaN checks)

In [ ]:
results = {}
for v in "ABCDE":
    lm.variant = v
    batch = lm.make_batch([examples[v]])
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    with torch.no_grad(), torch.autocast("cuda", dtype=lm.dtype):
        out = lm(batch, output_hidden_states=True)
    torch.cuda.synchronize()
    hs = out["hidden_states"]
    bad_layers = [i for i, h in enumerate(hs) if not torch.isfinite(h).all()]
    results[v] = {
        "loss": round(float(out["loss"]), 4),
        "sec": round(time.time() - t0, 2),
        "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
        "final_hidden": isfinite_report(out["hidden"]),
        "layers_with_nan_or_inf": bad_layers,
        "max_abs_hidden_per_layer": [round(float(h.float().abs().max()), 1) for h in hs][::5],
    }
    print(v, json.dumps(results[v]))
print("\nExpect: finite losses (pretrained model, ~0.5–1.5 nats on code); B–E slightly above A before training is normal.")

## Mask sanity check
For a late query token `q`: change a token `j` that the sliding mask hides from `q` → the output of the **first local layer's** attention at `q` must not change. Change a visible token → it must change. (Only the first layer is a strict test: in later layers hidden information can arrive indirectly through global layers or through visible tokens.)

In [ ]:
from scopesum.masks import MaskMeta, build_sliding_mask
from scopesum.model import capture_layer_attention, first_layer_of_type
from scopesum.sum_insertion import Example

LOCAL = first_layer_of_type(lm, "sliding_attention")
GLOBAL = first_layer_of_type(lm, "full_attention")
replacement = tok(" zebra", add_special_tokens=False).input_ids[0]

def attn_at(ex, ids, layer, q):
    e2 = Example.from_dict(ex.to_dict())
    e2.input_ids = ids
    with torch.autocast("cuda", dtype=lm.dtype):
        return capture_layer_attention(lm, lm.make_batch([e2]), layer)[0, q]

sanity = {}
for v in "ABCDE":
    lm.variant = v
    ex = examples[v]
    M = build_sliding_mask(v, MaskMeta.from_example(ex), window=lm.window, w_small=cfg.variant.w_small)
    kind = np.array(ex.kind)
    q = int(np.where(kind == 1)[0][-20])
    hidden = [j for j in range(1, q) if kind[j] == 1 and not M[q, j] and ex.input_ids[j] != replacement]
    visible = [j for j in range(1, q) if kind[j] == 1 and M[q, j] and ex.input_ids[j] != replacement]
    if not hidden:
        print(v, "nothing hidden from q (sequence shorter than the window?)"); continue
    j_h, j_v = hidden[len(hidden) // 2], visible[len(visible) // 2]
    base = list(ex.input_ids)
    ids_h = list(base); ids_h[j_h] = replacement
    ids_v = list(base); ids_v[j_v] = replacement
    ref_local = attn_at(ex, base, LOCAL, q)
    d_hidden = (attn_at(ex, ids_h, LOCAL, q) - ref_local).abs().max().item()
    d_visible = (attn_at(ex, ids_v, LOCAL, q) - ref_local).abs().max().item()
    d_global = (attn_at(ex, ids_h, GLOBAL, q) - attn_at(ex, base, GLOBAL, q)).abs().max().item()
    ok = d_hidden < 1e-3 and d_visible > 1e-3
    sanity[v] = dict(q=q, hidden_j=j_h, visible_j=j_v, visible_keys=int(M[q].sum()),
                     d_local_hidden=d_hidden, d_local_visible=d_visible, d_global_hidden=d_global, PASS=ok)
    print(v, sanity[v])
assert all(s["PASS"] for s in sanity.values()), "mask hook NOT working as expected"
print("\nMask hook OK: hidden tokens do not reach local layers; global layers still see them (d_global_hidden > 0).")

## Quick gradient check (one backward pass, variant C)

In [ ]:
lm.variant = "C"
lm.train()
lm.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
torch.cuda.reset_peak_memory_stats(); t0 = time.time()
with torch.autocast("cuda", dtype=lm.dtype):
    out = lm(lm.make_batch([examples["C"]]))
out["loss"].backward()
lora, extra = lm.trainable_parameters()
gn = [p.grad.norm().item() for p in lora if p.grad is not None]
print(f"loss {out['loss'].item():.4f} | LoRA params with grad: {len(gn)}/{len(lora)} | mean grad norm {np.mean(gn):.2e}")
print("<sum> embedding grad norm:", extra[0].grad.norm().item() if extra and extra[0].grad is not None else None)
print(f"train step: {time.time() - t0:.1f}s, peak {torch.cuda.max_memory_allocated() / 2**30:.2f} GB, "
      f"~{sum(k == 1 for k in examples['C'].kind) / (time.time() - t0):.0f} tokens/s (fwd+bwd, batch 1)")
lm.zero_grad(set_to_none=True)
lm.eval()